# CS1090a Final Project Milestone 2 

### Project  Title: Comparing Linear and Polynomial Models’ Ability to Predict Blood Lactate Levels with Metabolic and Biomechanical Indicators  

Group Members:
Sam Burgess (sburgess@college.harvard.edu)
Ethan Aidam (eaidam@college.harvard.edu)
Shay Li (sineadli@college.harvard.edu)
David Soucie(sirob7007@gmail.com)
Eva Harris (evaharris@college.harvard.edu)

## Table of Contents
* [Problem Statement](#problem-statement) 
* [Pre-Processing](#pre-processing)
    * [Import Libraries](#import-libraries)
    * [Data Loading](#data-loading) 
* [Data Description](#data-description)
    * [Dataset Qualities](#dataset-qualities)
        * [Entire Dataset](#qualities---entire-dataset)
        * [Organized by Day](#qualities---organized-by-day)
    * [Histograms](#Histograms)
    * [Correlation Plots](#correlation-plots)
        * [Entire Dataset](#correlations---entire-dataset)
        * [Organized by Day](#correlations---organized-by-day) 
    * [Clustering Plots](#clustering-plots)
    * [Findings Summary](#findings-summary)
* [Initial Model](#initial-model)
    * [Single-Variable Regression](#single-variable-regression)
    * [Multi-Variable Regression](#multi-variable-regression)

## Problem Statement

Understanding the factors influencing blood lactate threshold is vital for assessing athletic performance and designing personalized training. 
Traditional linear models may oversimplify the nonlinear relationships between biomechanical variables (e.g., gait dynamics) and metabolic measures (VO₂, EE). 

Our project explores whether polynomial regression better predicts lactate threshold, and uses time series modeling to capture trends in blood lactate across exercise intensities.

## Pre-Processing

### Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# pandas tricks for better display
pd.options.display.max_columns = 50  
pd.options.display.max_rows = 500     
pd.options.display.max_colwidth = 100
pd.options.display.precision = 3

### Data Loading

In [ ]:
day1_df = pd.read_csv("data/day1_lactate_challenge.csv")
day2_df = pd.read_csv("data/day2_lactate_challenge.csv")
day3_df = pd.read_csv("data/day3_lactate_challenge.csv")

day1_df["day"] = 1
day2_df["day"] = 2
day3_df["day"] = 3

df = pd.concat([day1_df, day2_df, day3_df], ignore_index=True)
df.head()

## Data Description

We will use a dataset of "Blood Lactate Concentration During Cycling High Intensity Interval Training". 
This data was collected from a single high performance athlete being tested for VO2 maximum using an indoor cycler across 3 days of testing, using various wearable devices and monitors. 
The data was entirely complete, with no null values. As a result, the only pre-processing step was combining each individual dataset and labeling each set of data with a "Day" value. Most values are floats, though a few values are integers; all are numeric.

The dataset includes the following 9 features, all of which with 8818 entries:
    - time: Second of test when measurement was taken. 
    - power: Cycling power output, measured in Watts. 
    - VO2: Oxygen consumption, measured in O2/min. 
    - Cadence: Speed. Measured in revolutions per minute of the cycle.
    - Lactate: Blood Lactate Concentration, measured in milliMols.
    - rf: Respiratory Frequency, measured in breaths per minute.
    - heart_rate: Heart Rate, measured in Beats per Minute.
    - saturation: Blood Oxygen Saturation, measured as a percentage.
    - Day: Added in pre-processing, indicates which day the data was collected.

In [ ]:
df.info()

### Dataset Qualities
#### Qualities - Entire Dataset

In [ ]:
pd.DataFrame({"Null count" : df.isnull().sum(), 
              "Mean" : df.mean(), 
              "Standard Deviation" : df.std(), 
              "Maximum" : df.max(), 
              "Minimum" : df.min()})

#### Qualities - Organized by Day
Day 1

In [ ]:
pd.DataFrame({"Null count" : day1_df.isnull().sum(), 
              "Mean" : day1_df.mean(), 
              "Standard Deviation" : day1_df.std(), 
              "Maximum" : day1_df.max(), 
              "Minimum" : day1_df.min()})

Day 2

In [ ]:
pd.DataFrame({"Null count" : day2_df.isnull().sum(), 
              "Mean" : day2_df.mean(), 
              "Standard Deviation" : day2_df.std(), 
              "Maximum" : day2_df.max(), 
              "Minimum" : day2_df.min()})

Day 3

In [ ]:
pd.DataFrame({"Null count" : day3_df.isnull().sum(), 
              "Mean" : day3_df.mean(), 
              "Standard Deviation" : day3_df.std(), 
              "Maximum" : day3_df.max(), 
              "Minimum" : day3_df.min()})

Some variation can be seen in the mean and standard deviation of each field on a per-day basis. However, unless these differences are also reflected in the correlations for each feature, it should not affect the results significantly.

However, what could affect results is the high discrepency between mean, variance, and standard deviation between the different values. As a result, a Scaler is used to normalize these values to insure that the differences do not bias the model or compromise potential conclusions. In order to preserve data consistency, the scaler is applied to the dataset with all 3 days, with all subsequent analysis on individual days using the data from the main, scaled dataframe.

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)
df = pd.DataFrame(scaler.fit_transform(df), columns = df.columns)
day1_df = df[df["day"] == -1.1398413637308449]
day2_df = df[df["day"] == 0.11969762691810124]
day3_df = df[df["day"] == 1.3792366175670472]

df.head()

### Histograms

In [ ]:
df.hist(figsize=(10, 6), bins = 10, edgecolor = "black")

plt.suptitle("Histogram of each Column", fontsize = 16)
plt.tight_layout()
plt.show()

No noteworthy concerns from the histogram, though some features do give insight on the nature of the performance. The athlete held a similar cadence and power level for the majority of the test, but with occasional bursts of higher levels of power and speed. This matches the typical process for VO2 max testing, with a default steady pace followed by bursts of max effort at specific intervals. 

### Correlation Plots

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(corr, cmap="coolwarm")

plt.title("Correlation Matrix", y=1.15)
plt.colorbar(cax)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="left")
plt.yticks(range(len(corr.columns)), corr.columns)

#### Correlations - Entire Dataset

In [ ]:
def display_corr(corr):
    corr_flat = (
        corr.unstack()                       # convert to Series with MultiIndex (row, column)
             .reset_index()                  # convert to columns
             .rename(columns={'level_0': 'Variable 1',
                              'level_1': 'Variable 2',
                              0: 'Correlation'})
    )
    corr_flat = corr_flat[corr_flat['Variable 1'] != corr_flat['Variable 2']]
    corr_flat = corr_flat.drop_duplicates(subset=['Correlation'])
    
    corr_flat = corr_flat.sort_values(by='Correlation', ascending=False)
    return corr_flat

display_corr(corr)

#### Correlations - Individual Days

Day 1

In [ ]:
display_corr(day1_df.corr())

Day 2

In [ ]:
display_corr(day2_df.corr())

Day 3

In [ ]:
display_corr(day3_df.corr())

Via the correlation matrix, we can see a few notably strong correlations, those being VO2 with heart rate and respiration, respiration with heart rate, and VO2 with cadence. A relatively strong negative correlation can also be seen with time and saturation.

However, we can also see significant variation in the correlations between specific features. This suggests that the athlete may have different performance circumstances across training sessions, potentially due to factors like sleep, fatigue, and lifestyle influence, all of which lack metrics in this dataset. 
Regardless, no significant correlations between the specific day and each feature can be seen. For this reason, a model trained using a dataset of the entire test set can be trusted to be reasonably accurate depiction of predictive ability, so long as it doesn't perform significantly weaker on any one day of the test. 


### Clustering Plots

In [ ]:
sns.clustermap(corr, cmap="coolwarm", annot=True)

plt.suptitle("Hierarchical Clustered Correlation Heatmap", y=1.02)
plt.show()

The clustermap shows the similar effects of the correlations showed, with 3 distinct clusters:
    - VO2, rf, and heart_rate
    - power, cadence, VO2, and heart rate
    - Saturation, lactate, and time


This suggests these features could be collinear, and thus only one of them should be included in the final models. 

## Findings Summary

In summary, we can find strong initial correlations between various features of the dataset which will be useful for constructing a series of regression tests. In addition, we have identified a few notably issues with the dataset, such as discrepencies in correlations between different days of testing, as well as instances of potential collinearity, which will need to be accounted for when evaluating the hypothesis.

## Initial Model 

For our initial model, we tested the predictive ability of several basic linear regression models, as well as a multi-variable linear model composed of the strongest performing features that did not exhibit collinearity  


### Single-Variable Regression

In [ ]:
def test_model(X, model, y = df["lactate"]):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return mean_squared_error(y_test, y_pred)

results_df = pd.DataFrame({
    "Feature" : df.columns,
    "Linear Regression MSE" : [test_model(feature[1].to_frame(), LinearRegression()) for feature in df.items()]
})
results_df = results_df[results_df["Feature"] != "lactate"]
results_df = results_df.sort_values(by = "Linear Regression MSE", ascending=True)
results_df

From the initial test, we see that the most predictive features are time, rf, and saturation, with further features having incredibly low predictive power, lower even than the day of the test itself. As a result, these 3 features will be chosen for an initial multi-variable linear regression

### Multi-Variable Regression

In [ ]:
print("Multivariable Regression with feature set [time, rf, saturation]: " + str(test_model(pd.DataFrame(df[["time", "rf", "saturation"]]), LinearRegression())))

While the multivariable model performed better, this initial test shows that a simple linear model has relatively low predictive power, suggesting an approach using more complex regressions may be useful for predicting lactate threshold.